# Model Training

Train ML models for price prediction.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import DEFAULT_TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

print("="*70)
print("MODEL TRAINING ON KAGGLE DATA")
print("="*70)
print(f"Training Period: {TRAIN_START} to {TRAIN_END}")
print(f"Test Period: {TEST_START} to {TEST_END}")
print("="*70)

MODEL TRAINING ON KAGGLE DATA
Training Period: 2015-01-01 to 2023-12-31
Test Period: 2024-01-01 to 2024-12-31


In [2]:
# Load and prepare data for first ticker
ticker = DEFAULT_TICKERS[0]
print(f"\nLoading {ticker}...")

raw_data = load_kaggle_data(ticker)
cleaned_data = clean_ohlcv_data(raw_data)

print(f"Cleaned data shape: {cleaned_data.shape}")
print(f"Date range: {cleaned_data.index[0]} to {cleaned_data.index[-1]}")

# Split by dates
train_data, test_data = split_data_by_date(cleaned_data)
print(f"\nTrain data: {train_data.shape}")
print(f"Test data: {test_data.shape}")


Loading NIFTY BANK...
2025-12-22 16:39:00 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-22 16:39:01 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-22 16:39:01 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-22 16:39:01 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-22 16:39:01 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-22 16:39:01 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-22 16:39:01 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-22 16:39:01 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-16 

## Check Available Data Intervals

In [3]:
%pip install seaborn --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Add features
def add_basic_features(data):
    df = data.copy()
    
    # Returns
    df['returns'] = df['Close'].pct_change()
    df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
    
    # Moving averages (use shorter windows to avoid too many NaNs)
    df['SMA_10'] = df['Close'].rolling(10).mean()
    df['SMA_20'] = df['Close'].rolling(20).mean()
    
    # Price range
    df['range'] = df['High'] - df['Low']
    df['range_pct'] = df['range'] / df['Close']
    
    # Volume features
    df['Volume_SMA_10'] = df['Volume'].rolling(10).mean()
    df['Volume_ratio'] = df['Volume'] / (df['Volume_SMA_10'] + 1e-8)  # Avoid division by zero
    
    # Volatility (shorter window)
    df['volatility_10'] = df['returns'].rolling(10).std()
    
    # RSI (simpler version)
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Target: 1 if next close > current close, else 0
    df['target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    
    # Forward fill NaNs from rolling calculations, then drop remaining NaNs
    df = df.fillna(method='bfill').fillna(method='ffill')
    df = df.dropna()
    
    # Remove last row (target is shifted)
    if len(df) > 0:
        df = df[:-1]
    
    return df

# Add features
train_with_features = add_basic_features(train_data)
test_with_features = add_basic_features(test_data)

print(f"Train data shape before features: {train_data.shape}")
print(f"Test data shape before features: {test_data.shape}")
print(f"\nTrain with features: {train_with_features.shape}")
print(f"Test with features: {test_with_features.shape}")

if len(train_with_features) > 0:
    print(f"Features: {[col for col in train_with_features.columns if col != 'target']}")
else:
    print("WARNING: No training data after feature engineering!")
    
if len(test_with_features) == 0:
    print("WARNING: No test data after feature engineering!")


Train with features: (0, 16)
Test with features: (0, 16)
Features: ['Open', 'High', 'Low', 'Close', 'Volume', 'returns', 'log_returns', 'SMA_20', 'SMA_50', 'range', 'range_pct', 'Volume_SMA_20', 'Volume_ratio', 'volatility_20', 'RSI']


In [5]:
# Prepare X and y
feature_cols = [col for col in train_with_features.columns if col not in ['target', 'Open', 'High', 'Low', 'Close', 'Volume']]

X_train = train_with_features[feature_cols]
y_train = train_with_features['target']

X_test = test_with_features[feature_cols]
y_test = test_with_features['target']

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Target distribution (train): {y_train.value_counts().to_dict()}")

X_train shape: (0, 10)
y_train shape: (0,)
Target distribution (train): {}


In [6]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled train mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(f"Scaled test mean: {X_test_scaled.mean():.4f}, std: {X_test_scaled.std():.4f}")

ValueError: Found array with 0 sample(s) (shape=(0, 10)) while a minimum of 1 is required by StandardScaler.

In [ ]:
# Train Random Forest
print("\nTraining RandomForest classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_train = rf_model.predict(X_train_scaled)
y_pred_test = rf_model.predict(X_test_scaled)

train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_test))

## Improve Model Accuracy

Try these optimization strategies to increase accuracy:

In [ ]:
# Train models for all tickers
results = {}

print("\n" + "="*70)
print("TRAINING ALL TICKERS")
print("="*70)

for ticker in DEFAULT_TICKERS:
    print(f"\nTraining {ticker}...", end=" ")
    
    try:
        # Load and prepare
        raw_data = load_kaggle_data(ticker)
        cleaned = clean_ohlcv_data(raw_data)
        train, test = split_data_by_date(cleaned)
        
        # Features
        train_f = add_basic_features(train)
        test_f = add_basic_features(test)
        
        if len(train_f) == 0 or len(test_f) == 0:
            print("SKIPPED (no data)")
            continue
        
        # Train model
        X_train = train_f[feature_cols]
        y_train = train_f['target']
        X_test = test_f[feature_cols]
        y_test = test_f['target']
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
        model.fit(X_train_scaled, y_train)
        
        test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
        
        results[ticker] = {
            'model': model,
            'scaler': scaler,
            'features': feature_cols,
            'train_samples': len(X_train),
            'test_samples': len(X_test),
            'test_accuracy': test_acc
        }
        
        print(f"✅ Accuracy: {test_acc:.4f} | Train: {len(X_train)} | Test: {len(X_test)}")
        
    except Exception as e:
        print(f"❌ Error: {str(e)[:50]}")

print("\n" + "="*70)
print(f"Successfully trained {len(results)} models")
print("="*70)

In [ ]:
# Option 2: Use GridSearchCV to find best XGBoost hyperparameters
best_xgb, best_cv_score = tune_xgboost_hyperparameters(X_train, y_train)

# Then train stacked model with tuned hyperparameters
model_tuned, tuned_accuracy = train_tuned_stacked_model(X_train, y_train, X_test, y_test)
y_pred_tuned = model_tuned.predict(X_test)

print(f"\n--- Accuracy Comparison ---")
print(f"Original Model:  {acc_original:.4f}")
print(f"Improved Model:  {acc_improved:.4f}")
print(f"Tuned Model:     {tuned_accuracy:.4f}")

# Evaluate tuned model
metrics_tuned = evaluate_classification(y_test, y_pred_tuned, model_name='Tuned Stacked (RF + XGB + GB)')

### Other Strategies to Improve Accuracy:

1. **Feature Engineering**: Create new features or remove irrelevant ones
2. **Data Preprocessing**: Better handling of outliers, normalization
3. **Class Imbalance**: Use SMOTE or adjust class weights further
4. **Ensemble**: Add more diverse base learners (SVM, Neural Networks)
5. **Target Engineering**: Ensure target variable is well-defined
6. **Feature Scaling**: Try different scaling methods